In [1]:
!pip install nlpaug -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 7.8 MB/s eta 0:00:00


In [2]:
# %%writefile biobertpt_contrastive_finetuning.py

# =============================================================================
# Fine-Tuning pucpr/biobertpt-all with Contrastive Learning
# Refactored: HuggingFace Datasets API + scikit-learn APIs throughout
# Target: Kaggle (T4 x2 / P100)
# =============================================================================

# --- Install Dependencies (first Kaggle cell) ---
# !pip install sentence-transformers datasets nlpaug scikit-learn -q
# !pip install transformers -q
#
# nlpaug downloads neuralmind/bert-base-portuguese-cased (~400 MB, HF-cached).
# Set aug_strategy="rule_based" in main() to skip the download entirely.

import os
import random
import math
import re
import warnings
import numpy as np
import pandas as pd
import torch
from collections import Counter
from typing import List, Dict, Any

# --- HuggingFace ---
from datasets import Dataset, DatasetDict, concatenate_datasets

# --- scikit-learn ---
from sklearn.utils import resample as sklearn_resample
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# --- Sentence Transformers ---
from sentence_transformers import SentenceTransformer, InputExample, losses, models, evaluation
from torch.utils.data import DataLoader

import transformers

# Monkey patch to fix nlpaug bug with modern transformers library
transformers.PreTrainedTokenizerBase._convert_token_to_id = transformers.PreTrainedTokenizerBase.convert_tokens_to_ids
transformers.PreTrainedTokenizerFast._convert_token_to_id = lambda self, token: self.convert_tokens_to_ids(token)

# --- nlpaug (optional, graceful fallback) ---
try:
    import nlpaug.augmenter.word as naw
    NLPAUG_AVAILABLE = True
except ImportError:
    NLPAUG_AVAILABLE = False
    print("[WARN] nlpaug not found — rule-based augmentation will be used.")
    print("       Install with:  !pip install nlpaug -q")


# =============================================================================
# SECTION 0: Reproducibility & Multi-GPU Configuration
# =============================================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def configure_gpu(mode: str = "dp") -> str:
    """
    Configures the GPU environment and returns the device string.

    Background: Why the DataParallel warning fires
    -----------------------------------------------
    sentence-transformers' model.fit() automatically wraps the model in
    torch.nn.DataParallel (DP) when it detects > 1 GPU.  DP replicates
    the model on each GPU and splits each batch across them, but it does
    so through a single Python process — this causes GIL contention and
    heavy GPU-0 memory usage.

    The library prefers DistributedDataParallel (DDP) because DDP spawns
    one process per GPU, eliminating GIL overhead.  However, DDP requires
    launching the *entire script* with torchrun, which cannot be done
    inside a Kaggle notebook cell.

    Available modes
    ---------------
    "single"  — Pin to GPU 0 only (CUDA_VISIBLE_DEVICES=0).
                No multi-GPU, no warning.  Safe default for notebooks.
                Effective GPU: T4 or P100 (one card).

    "dp"      — Allow DataParallel on all visible GPUs.
                Warning is suppressed here so logs stay clean.
                Use when you want both T4s without rewriting as a script.

    "ddp"     — Intended for script-mode only (not notebook cells).
                This function sets mode back to 'single' with a reminder.
                To truly use DDP, run the script from a terminal:

                    torchrun --nproc_per_node=2 biobertpt_contrastive_finetuning.py

                DDP gives ~1.8× speedup over DP on T4 x2 for BERT training.

    Parameters
    ----------
    mode : "single" | "dp" | "ddp"

    Returns
    -------
    device string ("cuda" or "cpu") usable by SentenceTransformer.
    """
    n_gpus = torch.cuda.device_count()

    if not torch.cuda.is_available() or n_gpus == 0:
        print("[GPU] No CUDA device found — using CPU.")
        return "cpu"

    if mode == "single" or n_gpus == 1:
        # Restrict to GPU 0 — eliminates DataParallel path entirely
        os.environ["CUDA_VISIBLE_DEVICES"] = "0"
        device = "cuda"
        print(f"[GPU] Single-GPU mode  → GPU 0: {torch.cuda.get_device_name(0)}")

    elif mode == "dp":
        # Suppress ONLY the DP-vs-DDP informational warning (not errors)
        warnings.filterwarnings(
            "ignore",
            message=".*DataParallel.*DistributedDataParallel.*",
            category=UserWarning,
        )
        device = "cuda"
        print(f"[GPU] DataParallel mode → {n_gpus} GPUs detected. "
              "DP warning suppressed.")
        for i in range(n_gpus):
            print(f"       GPU {i}: {torch.cuda.get_device_name(i)}")
        print("[GPU] Tip: for faster training run as a script with:")
        print("       torchrun --nproc_per_node=2 "
              "biobertpt_contrastive_finetuning.py")

    elif mode == "ddp":
        # DDP requires torchrun — fall back to single in notebook context
        print("[GPU] DDP mode requested but torchrun is required.")
        print("[GPU] Falling back to single-GPU for notebook execution.")
        print("[GPU] To use DDP, run from a terminal:")
        print("       torchrun --nproc_per_node=2 "
              "biobertpt_contrastive_finetuning.py")
        os.environ["CUDA_VISIBLE_DEVICES"] = "0"
        device = "cuda"

    else:
        raise ValueError(f"Unknown GPU mode '{mode}'. "
                         "Choose 'single', 'dp', or 'ddp'.")

    return device


# ── Initialise GPU (change mode here) ────────────────────────────────────────
# "single" → one GPU, no warning  (recommended for Kaggle notebooks)
# "dp"     → both GPUs, warning suppressed
# "ddp"    → reminder to use torchrun, falls back to single in notebook
DEVICE = configure_gpu(mode="dp")

# Extreme minority classes that receive augmented up-sampling
MINORITY_CLASSES_FOR_AUG = {4, 5, 6}


# =============================================================================
# SECTION 1: Mock Dataset  →  HuggingFace Dataset
# Replace create_mock_dataset() with Dataset.from_pandas(your_df) for real data
# =============================================================================

def create_mock_dataset() -> Dataset:
    """
    Builds a HuggingFace Dataset that mirrors the real class distribution.
    Features: text (string), target (int64).

    Class distribution (mock):
        Class 2 → 15,968  (dominant)
        Class 3 →    713
        Class 1 →    693
        Class 0 →    610
        Class 4 →    214
        Class 6 →     45
        Class 5 →     29

    HF Dataset.from_dict() is used directly — no intermediate pandas step.
    """
    templates = {
        0: "Paciente apresenta resultado negativo para malignidade, categoria BI-RADS 0.",
        1: "Inconclusivo, necessita de exames adicionais, BI-RADS 1.",
        2: "Achado benigno confirmado, sem necessidade de intervenção, BI-RADS 2.",
        3: "Provavelmente benigno, seguimento de curto prazo recomendado, BI-RADS 3.",
        4: "Achado suspeito, biópsia indicada, BI-RADS 4.",
        5: "Achado altamente sugestivo de malignidade, BI-RADS 5.",
        6: "Malignidade confirmada por biópsia prévia, BI-RADS 6.",
    }
    counts = {0: 610, 1: 693, 2: 15968, 3: 713, 4: 214, 5: 29, 6: 45}

    texts, targets = [], []
    for cls, n in counts.items():
        for i in range(n):
            texts.append(
                f"Laudo #{i+1}: {templates[cls]} Exame ID {random.randint(1000, 9999)}."
            )
            targets.append(cls)

    # Build HF Dataset directly from dict — no pandas required
    ds = Dataset.from_dict({"text": texts, "target": targets})
    ds = ds.shuffle(seed=SEED)

    dist = Counter(ds["target"])
    print(f"[DATA] HF Dataset created → {len(ds):,} rows")
    for c in sorted(dist):
        print(f"       Class {c}: {dist[c]:>6,}")
    print()
    return ds


# =============================================================================
# SECTION 2b: Augmentation Engine
# (defined before 2a so balance_dataset() can call it)
# =============================================================================

def _build_augmenter(strategy: str, aug_p: float = 0.15, seed: int = SEED):
    """
    Factory for the nlpaug ContextualWordEmbsAug augmenter, or None.

    Model: neuralmind/bert-base-portuguese-cased
        General Portuguese BERT (2.5B tokens, Neuralmind/UNICAMP).
        Wider vocabulary than BioBERTpt → more varied lexical substitutions.
        Swap model_path to "pucpr/biobertpt-all" for domain-specific replacements.

    aug_p=0.15: replaces ~15% of tokens per sentence.
        < 0.10 → too little variation
        > 0.30 → risks flipping clinical meaning
    """
    np.random.seed(seed)

    if strategy == "contextual":
        if not NLPAUG_AVAILABLE:
            print("[AUG] nlpaug unavailable → switching to rule_based.")
            return None
        print("[AUG] Loading ContextualWordEmbsAug "
              "(neuralmind/bert-base-portuguese-cased) …")

        aug = naw.ContextualWordEmbsAug(
            model_path="neuralmind/bert-base-portuguese-cased",
            action="substitute",
            aug_p=aug_p,
            aug_min=1,
            device=DEVICE,
            batch_size=8,
        )
        print("[AUG] Augmenter ready.\n")
        return aug
    return None  # rule_based / none → no nlpaug object needed


def _rule_based_augment(text: str, rng: random.Random) -> str:
    """
    Pure-Python fallback augmentation (zero extra downloads).
    Randomly applies ONE of three lightweight transforms:

    0 → Numeral perturbation  : ±1 on any isolated integer
    1 → Punctuation restructure: remove a mid-sentence period
    2 → Abbreviation swap      : BI-RADS↔BIRADS, biópsia↔biopsia, etc.
    """
    choice = rng.randint(0, 2)

    if choice == 0:
        return re.sub(
            r"\b\d+\b",
            lambda m: str(max(0, int(m.group()) + rng.choice([-1, 1]))),
            text,
        )
    elif choice == 1:
        parts = text.split(". ")
        if len(parts) > 1:
            idx = rng.randint(0, len(parts) - 2)
            parts[idx] = parts[idx].rstrip(".")
        return ". ".join(parts)
    else:
        abbr_map = [
            ("BI-RADS", "BIRADS"), ("biópsia", "biopsia"),
            ("prévia", "anterior"), ("seguimento", "acompanhamento"),
            ("achado", "finding"),
        ]
        rng.shuffle(abbr_map)
        for src, tgt in abbr_map:
            if src.lower() in text.lower():
                return re.sub(re.escape(src), tgt, text, count=1,
                              flags=re.IGNORECASE)
        # Fallback within fallback: numeral perturbation
        return re.sub(
            r"\b\d+\b",
            lambda m: str(max(0, int(m.group()) + rng.choice([-1, 1]))),
            text,
        )


def _make_aug_map_fn(augmenter, aug_strategy: str, rng: random.Random):
    """
    Returns a HuggingFace Dataset .map()-compatible row function.

    Using .map() instead of a manual Python loop gives us:
    - Built-in tqdm progress bar
    - Optional HF caching (disabled here: stochastic augmentation)
    - Consistent interface that works identically on CPU and GPU

    The returned function is row-level (batched=False) for per-sample
    error handling — a single failed nlpaug call falls back to rule_based
    without aborting the entire batch.
    """
    def augment_row(example: Dict[str, Any]) -> Dict[str, Any]:
        text = example["text"]
        if aug_strategy == "contextual" and augmenter is not None:
            try:
                result = augmenter.augment([text])
                # nlpaug may return list-of-list or flat list — normalise:
                if result and isinstance(result[0], list):
                    aug_text = result[0][0] if result[0] else text
                else:
                    aug_text = result[0] if result else text
            except Exception:
                aug_text = _rule_based_augment(text, rng)
        else:
            aug_text = _rule_based_augment(text, rng)
        return {"text": aug_text, "target": example["target"]}

    return augment_row


# =============================================================================
# SECTION 2a: Data Balancing  (HF Datasets API + sklearn.utils.resample)
# =============================================================================

def balance_dataset(
    dataset: Dataset,
    majority_class: int = 2,
    majority_cap: int = 1000,
    min_samples_per_class: int = 150,
    aug_strategy: str = "contextual",
    aug_batch_size: int = 8,
    seed: int = SEED,
) -> Dataset:
    """
    Rebalances a HuggingFace Dataset using HF Dataset ops + sklearn.

    Per-class strategy
    ------------------
    Majority class (2)          → sklearn_resample(replace=False) down to majority_cap
    Extreme minority (4, 5, 6)  → originals kept + HF .map() augmentation for delta rows
    Near-minority (0, 1, 3)     → sklearn_resample(replace=True) bootstrap up to floor
    Others                      → unchanged (returned via HF .filter())

    HF Datasets API used
    --------------------
    dataset.filter()            class-specific subsets
    dataset.select()            index-based row selection after sklearn gives indices
    dataset.map()               row-wise augmentation with progress bar
    dataset.shuffle()           pre-resample shuffle for randomness
    concatenate_datasets()      reassemble balanced class parts

    sklearn APIs used
    -----------------
    sklearn.utils.resample(replace=False)  stratified down-sampling
    sklearn.utils.resample(replace=True)   bootstrap up-sampling
    """
    rng = random.Random(seed)
    augmenter = _build_augmenter(aug_strategy, seed=seed)
    aug_map_fn = _make_aug_map_fn(augmenter, aug_strategy, rng)

    all_classes = sorted(set(dataset["target"]))
    parts: List[Dataset] = []

    for cls in all_classes:
        # ── Isolate class via HF Dataset.filter() ───────────────────────
        cls_ds: Dataset = dataset.filter(
            lambda x, c=cls: x["target"] == c,
            desc=f"Filtering class {cls}",
        )
        n = len(cls_ds)

        if cls == majority_class:
            # ── sklearn_resample (replace=False) — no duplicates ─────────
            down_idx = sklearn_resample(
                range(n),
                replace=False,
                n_samples=min(n, majority_cap),
                random_state=seed,
            )
            sampled = cls_ds.select(sorted(down_idx))
            print(f"[BALANCE] Class {cls}: {n:>6} → {len(sampled):>6}  "
                  f"(sklearn_resample DOWN, replace=False)")

        elif n < min_samples_per_class and cls in MINORITY_CLASSES_FOR_AUG:
            # ── Augmented up-sampling for extreme minority ────────────────
            n_needed = min_samples_per_class - n

            # sklearn_resample(replace=True): bootstrap — pick source rows
            source_idx = sklearn_resample(
                range(n),
                replace=True,
                n_samples=n_needed,
                random_state=seed,
            )
            sources_ds = cls_ds.select(list(source_idx))

            # HF .map() applies augmentation with a built-in progress bar
            aug_ds: Dataset = sources_ds.map(
                aug_map_fn,
                desc=f"Augmenting class {cls} (+{n_needed})",
                load_from_cache_file=False,
            )
            sampled = concatenate_datasets([cls_ds, aug_ds])
            print(f"[BALANCE] Class {cls}: {n:>6} → {len(sampled):>6}  "
                  f"(AUG '{aug_strategy}' +{n_needed} rows via HF .map())")

        elif n < min_samples_per_class:
            # ── sklearn_resample(replace=True) — bootstrap for near-minority
            up_idx = sklearn_resample(
                range(n),
                replace=True,
                n_samples=min_samples_per_class,
                random_state=seed,
            )
            sampled = cls_ds.select(list(up_idx))
            print(f"[BALANCE] Class {cls}: {n:>6} → {len(sampled):>6}  "
                  f"(sklearn_resample UP, replace=True)")

        else:
            sampled = cls_ds
            print(f"[BALANCE] Class {cls}: {n:>6} → {len(sampled):>6}  "
                  f"(unchanged)")

        parts.append(sampled)

    # ── Reassemble via HF concatenate_datasets + shuffle ────────────────
    balanced: Dataset = concatenate_datasets(parts).shuffle(seed=seed)
    dist = Counter(balanced["target"])
    print(f"\n[BALANCE] Balanced dataset → {len(balanced):,} rows")
    for c in sorted(dist):
        print(f"          Class {c}: {dist[c]:>6,}")
    print()
    return balanced


# =============================================================================
# SECTION 3: Contrastive Pair Generation  (uses LabelEncoder + HF .to_pandas)
# =============================================================================

def generate_contrastive_pairs(
    dataset: Dataset,
    n_pairs: int = 20_000,
    positive_ratio: float = 0.50,
    seed: int = SEED,
) -> List[InputExample]:
    """
    Generates InputExample pairs for ContrastiveLoss.
        label 1.0 → positive pair (same class)
        label 0.0 → negative pair (different classes)

    HF Datasets API used
    --------------------
    dataset.to_pandas()         fast single-pass conversion for groupby

    sklearn APIs used
    -----------------
    LabelEncoder                normalises class indices to be contiguous
                                (safe if a class disappears after balancing)

    Positive pair sampling uses inverse-frequency class weights so extreme
    minority classes appear proportionally rather than being overwhelmed.
    """
    rng = random.Random(seed)

    # sklearn LabelEncoder — safe even if some class labels are non-contiguous
    le = LabelEncoder()
    le.fit(dataset["target"])
    classes: List[int] = le.classes_.tolist()

    # HF .to_pandas() for efficient per-class groupby
    df = dataset.to_pandas()
    class_to_texts: Dict[int, List[str]] = {
        int(c): df[df["target"] == c]["text"].tolist() for c in classes
    }

    # Inverse-frequency class weights for positive pair minority oversampling
    sizes = np.array([len(class_to_texts[c]) for c in classes], dtype=float)
    weights = (1.0 / sizes) / (1.0 / sizes).sum()

    n_pos = int(n_pairs * positive_ratio)
    n_neg = n_pairs - n_pos
    examples: List[InputExample] = []

    # ── Positive pairs ───────────────────────────────────────────────────
    for _ in range(n_pos):
        cls = rng.choices(classes, weights=weights.tolist(), k=1)[0]
        pool = class_to_texts[cls]
        a, b = rng.sample(pool, 2) if len(pool) >= 2 else (pool[0], pool[0])
        examples.append(InputExample(texts=[a, b], label=1.0))

    # ── Negative pairs ───────────────────────────────────────────────────
    for _ in range(n_neg):
        ca, cb = rng.sample(classes, 2)
        examples.append(InputExample(
            texts=[rng.choice(class_to_texts[ca]),
                   rng.choice(class_to_texts[cb])],
            label=0.0,
        ))

    rng.shuffle(examples)
    pos = sum(1 for e in examples if e.label == 1.0)
    print(f"[PAIRS] {len(examples):,} pairs  "
          f"(+{pos:,} positive / -{len(examples)-pos:,} negative)\n")
    return examples


# =============================================================================
# SECTION 4: Model
# =============================================================================

def load_model(
    model_name: str = "pucpr/biobertpt-all",
    max_seq_length: int = 512,
) -> SentenceTransformer:
    """
    Explicitly constructs a SentenceTransformer from pucpr/biobertpt-all.

    Why explicit construction?
    --------------------------
    pucpr/biobertpt-all is a plain HuggingFace BERT model without a
    sentence-transformers config file on the Hub. Passing it directly to
    SentenceTransformer() triggers:
        WARNING: No sentence-transformers model found … Creating a new one
                 with mean pooling.
    The library still produces the correct architecture, but the warning is
    misleading. Explicitly building the modules:
        1. Silences the warning entirely.
        2. Makes the pooling strategy self-documented and easy to change
           (swap pooling_mode to "cls" or "max" without touching anything else).
        3. Lets us set max_seq_length precisely (BERT-base supports up to 512).

    Architecture
    ------------
    models.Transformer  →  per-token hidden states  (768-d × seq_len)
    models.Pooling      →  mean over token dimension  (768-d vector)
    """
    from sentence_transformers import models

    print(f"[MODEL] Loading '{model_name}' (explicit Transformer + MeanPooling) …")

    # ── 1. Transformer encoder ────────────────────────────────────────────
    transformer = models.Transformer(
        model_name,
        max_seq_length=max_seq_length,
    )

    # ── 2. Mean pooling layer ─────────────────────────────────────────────
    # pooling_mode_mean_tokens=True  → average all non-padding token vectors
    # Change to pooling_mode_cls_token=True for [CLS]-based embeddings
    pooling = models.Pooling(
        transformer.get_word_embedding_dimension(),
        pooling_mode_mean_tokens=True,
        pooling_mode_cls_token=False,
        pooling_mode_max_tokens=False,
    )

    # ── 3. Assemble SentenceTransformer ──────────────────────────────────
    model = SentenceTransformer(
        modules=[transformer, pooling],
        device=DEVICE,
    )

    print(f"[MODEL] Embedding dim : {model.get_sentence_embedding_dimension()}")
    print(f"[MODEL] Max seq length: {model.max_seq_length}\n")
    return model


# =============================================================================
# SECTION 5: DataLoader + Training
# =============================================================================

def build_dataloader(
    examples: List[InputExample],
    batch_size: int = 32,
    shuffle: bool = True,
) -> DataLoader:
    loader = DataLoader(examples, shuffle=shuffle, batch_size=batch_size)
    print(f"[LOADER] {len(examples):,} pairs | "
          f"batch={batch_size} | steps/epoch≈{len(loader):,}\n")
    return loader


def train_model(
    model: SentenceTransformer,
    train_dataloader: DataLoader,
    val_dataloader: DataLoader,
    epochs: int = 3,
    warmup_fraction: float = 0.10,
    output_path: str = "./biobertpt_contrastive",
    use_online_contrastive: bool = False,
) -> SentenceTransformer:
    """
    Fine-tunes with ContrastiveLoss or OnlineContrastiveLoss.

    Hyperparameters (BERT-base standard)
    -------------------------------------
    LR=2e-5, batch=32, epochs=4, warmup=10%, weight_decay=0.01, AMP=True
    """
    if use_online_contrastive:
        # loss_fn = losses.OnlineContrastiveLoss(model=model)
        loss_fn = losses.OnlineContrastiveLoss(
            model=model, 
            distance_metric=losses.SiameseDistanceMetric.COSINE_DISTANCE, 
            margin=0.6
        )
        print("[TRAIN] Loss: OnlineContrastiveLoss (hard-pair mining)")
    else:
        loss_fn = losses.ContrastiveLoss(model=model)
        print("[TRAIN] Loss: ContrastiveLoss (pre-formed pairs)")

    total   = len(train_dataloader) * epochs
    warmup  = int(total * warmup_fraction)
    print(f"[TRAIN] Steps={total:,}  Warmup={warmup:,}  "
          f"Epochs={epochs}  → '{output_path}'\n")
    
    evaluator = evaluation.BinaryClassificationEvaluator.from_input_examples(val_dataloader.dataset, name='birads-val')
    model.fit(
        train_objectives=[(train_dataloader, loss_fn)],
        evaluator=evaluator,
        evaluation_steps=100, 
        epochs=epochs,
        warmup_steps=warmup,
        optimizer_params={"lr": 2e-5},
        weight_decay=0.01,
        output_path=output_path,
        show_progress_bar=True,
        save_best_model=True,
        use_amp=True,               # FP16 — ~2× speedup on T4/P100
    )
    print(f"[TRAIN] Done. Saved to '{output_path}'\n")
    return model


# =============================================================================
# SECTION 6: Evaluation  (HF Dataset.train_test_split + sklearn metrics)
# =============================================================================

def evaluate_embeddings(
    model: SentenceTransformer,
    train_dataset: Dataset,
    val_dataset: Dataset,
    test_size: float = 0.20,
    seed: int = SEED,
) -> None:
    """
    Evaluates embedding quality with a sklearn k-NN probe classifier.

    Why k-NN?
    ---------
    k-NN cannot learn a complex decision boundary — it is the fairest probe
    for whether the embedding space itself is well-organised. A linear SVC
    could compensate for a poor embedding with its own learned weights.

    HF Datasets API used
    --------------------
    dataset.train_test_split(stratify_by_column="target")
        Native stratified splitting — mirrors sklearn's stratify= parameter.

    sklearn APIs used
    -----------------
    LabelEncoder                    normalise class labels
    KNeighborsClassifier(cosine)    embedding-space probe
    classification_report           per-class precision/recall/F1
    confusion_matrix                class confusion heatmap
    f1_score(average="macro")       imbalance-robust summary metric
    """
    print("\n" + "=" * 64)
    print("EVALUATION: sklearn k-NN Probe on Embedding Space")
    print("=" * 64)

    # ── Stratified split via HF Dataset API ─────────────────────────────
    # split: DatasetDict = dataset.train_test_split(
    #     test_size=test_size,
    #     seed=seed,
    #     stratify_by_column="target",   # HF preserves class proportions
    # )
    # train_dataset, val_dataset = split["train"], split["test"]
    print(f"[EVAL] Train: {len(train_dataset):,}  |  Test: {len(val_dataset):,}")

    # ── Encode embeddings ────────────────────────────────────────────────
    print("[EVAL] Encoding embeddings …")
    X_train = model.encode(train_dataset["text"], batch_size=64,
                           show_progress_bar=True, convert_to_numpy=True)
    X_test  = model.encode(val_dataset["text"],  batch_size=64,
                           show_progress_bar=True, convert_to_numpy=True)

    y_train = np.array(train_dataset["target"])
    y_test  = np.array(val_dataset["target"])

    # ── sklearn LabelEncoder ─────────────────────────────────────────────
    le = LabelEncoder()
    le.fit(y_train)
    y_tr_enc = le.transform(y_train)
    y_te_enc = le.transform(y_test)
    class_names = [f"Class {c}" for c in le.classes_]

    # ── k-NN probe (cosine distance) ─────────────────────────────────────
    knn = KNeighborsClassifier(n_neighbors=5, metric="cosine", n_jobs=-1)
    knn.fit(X_train, y_tr_enc)
    y_pred = knn.predict(X_test)

    # ── sklearn metrics ──────────────────────────────────────────────────
    macro_f1 = f1_score(y_te_enc, y_pred, average="macro", zero_division=0)
    print(f"\n[EVAL] Macro F1 (k-NN, cosine): {macro_f1:.4f}\n")
    print(classification_report(y_te_enc, y_pred,
                                target_names=class_names, zero_division=0))
    cm = confusion_matrix(y_te_enc, y_pred)
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    print("Confusion Matrix (rows=true, cols=pred):")
    print(cm_df.to_string())
    print("=" * 64 + "\n")


# =============================================================================
# SECTION 7: Cosine Sanity Check
# =============================================================================

def sanity_check(model: SentenceTransformer, dataset: Dataset) -> None:
    """
    Picks one text per class via HF .to_pandas() groupby and prints
    pairwise cosine similarities. Same-class → ~1.0, cross-class → ~0.0.
    """
    from sentence_transformers import util

    df = dataset.to_pandas().groupby("target").first().reset_index()
    texts  = df["text"].tolist()
    labels = df["target"].tolist()
    embs   = model.encode(texts, convert_to_tensor=True, show_progress_bar=False)

    print("\n" + "=" * 64)
    print("SANITY CHECK: Pairwise Cosine Similarity")
    print("=" * 64)
    for i in range(len(texts)):
        for j in range(i + 1, len(texts)):
            sim  = util.cos_sim(embs[i], embs[j]).item()
            kind = "SAME" if labels[i] == labels[j] else "DIFF"
            print(f"  [{kind}] Class {labels[i]} ↔ Class {labels[j]}  |  "
                  f"cosine = {sim:.4f}")
    print("=" * 64 + "\n")


# =============================================================================
# SECTION 8: Main Pipeline
# =============================================================================

def main():
    # ── 8.1  Data (HuggingFace Dataset) ─────────────────────────────────
    # dataset = create_mock_dataset()
    # Real data:
    df = pd.read_csv("/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv")
    dataset = Dataset.from_pandas(df[["report", "target"]])
    dataset = dataset.rename_column("report", "text")
    dataset = dataset.class_encode_column("target")
    dataset = dataset.shuffle(42)

    split_ds = dataset.train_test_split(test_size=0.15, seed=42, stratify_by_column="target")
    train_dataset = split_ds["train"]
    val_dataset = split_ds["test"]

    # ── 8.2  Balance + Augment (HF ops + sklearn_resample) ───────────────
    # aug_strategy: "contextual" | "rule_based" | "none"
    # balanced_ds = balance_dataset(
    #     dataset,
    #     majority_class=2,
    #     majority_cap=1000,
    #     min_samples_per_class=150,
    #     aug_strategy="contextual",     # ← "rule_based" if offline
    #     aug_batch_size=8,
    # )
    balanced_train_dataset = balance_dataset(
        train_dataset,
        majority_class=2,
        majority_cap=1500,
        min_samples_per_class=200,
        aug_strategy="contextual",     # ← "rule_based" if offline
        aug_batch_size=8,
    )

    # ── 8.3  Pair generation (LabelEncoder inside) ───────────────────────
    # examples = generate_contrastive_pairs(balanced_ds, n_pairs=20_000)
    train_examples = generate_contrastive_pairs(balanced_train_dataset, n_pairs=20_000)
    val_examples = generate_contrastive_pairs(val_dataset, n_pairs=3_000)

    # ── 8.4  DataLoader ──────────────────────────────────────────────────
    # train_loader = build_dataloader(examples, batch_size=16)
    train_loader = build_dataloader(train_examples, batch_size=16)
    val_loader = build_dataloader(val_examples, batch_size=16)

    # ── 8.5  Load model ──────────────────────────────────────────────────
    model = load_model("pucpr/biobertpt-all")

    # ── 8.6  Fine-tune ───────────────────────────────────────────────────
    model = train_model(
        model, train_loader, val_loader,
        epochs=4,
        output_path="./biobertpt_contrastive",
        use_online_contrastive=True,  # True → OnlineContrastiveLoss
    )

    # ── Alternatively, Load Fine-tuned Model ───────────────────────────────────────────────────
    # model = SentenceTransformer("/kaggle/input/models/lokeshvns/custom-embedding-model/transformers/default/1/biobertpt_contrastive")

    # ── 8.7  Evaluation (HF train_test_split + sklearn metrics) ──────────
    evaluate_embeddings(model, balanced_train_dataset, val_dataset)

    # ── 8.8  Sanity check ────────────────────────────────────────────────
    sanity_check(model, balanced_train_dataset)

    # # ── 8.9  Export embeddings for downstream use (e.g. classifier head) ─
    # all_emb = model.encode(
    #     balanced_ds["text"], batch_size=64,
    #     show_progress_bar=True, convert_to_numpy=True,
    # )
    # print(f"[ENCODE] Full embedding matrix: {all_emb.shape}")
    # np.save("embeddings.npy", all_emb)


if __name__ == "__main__":
    main()

[GPU] DataParallel mode → 2 GPUs detected. DP warning suppressed.
       GPU 0: Tesla T4
       GPU 1: Tesla T4
[GPU] Tip: for faster training run as a script with:
       torchrun --nproc_per_node=2 biobertpt_contrastive_finetuning.py


Stringifying the column:   0%|          | 0/18272 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/18272 [00:00<?, ? examples/s]

[AUG] Loading ContextualWordEmbsAug (neuralmind/bert-base-portuguese-cased) …


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[TRAIN] Done. Saved to './biobertpt_contrastive'


EVALUATION: sklearn k-NN Probe on Embedding Space
[EVAL] Train: 3,813  |  Test: 2,741
[EVAL] Encoding embeddings …


Batches:   0%|          | 0/60 [00:00<?, ?it/s]

Batches:   0%|          | 0/43 [00:00<?, ?it/s]


[EVAL] Macro F1 (k-NN, cosine): 0.6901

              precision    recall  f1-score   support

     Class 0       0.62      0.87      0.72        92
     Class 1       0.87      0.99      0.92       104
     Class 2       1.00      0.92      0.95      2395
     Class 3       0.34      0.78      0.47       107
     Class 4       0.68      0.59      0.63        32
     Class 5       0.50      0.50      0.50         4
     Class 6       0.56      0.71      0.62         7

    accuracy                           0.91      2741
   macro avg       0.65      0.77      0.69      2741
weighted avg       0.95      0.91      0.92      2741

Confusion Matrix (rows=true, cols=pred):
         Class 0  Class 1  Class 2  Class 3  Class 4  Class 5  Class 6
Class 0       80        0        2        9        1        0        0
Class 1        0      103        1        0        0        0        0
Class 2       33       15     2195      147        1        0        4
Class 3       16        1        4   

In [3]:
# !OMP_NUM_THREADS=1 torchrun --nproc_per_node=2 --master_port=12345 biobertpt_contrastive_finetuning.py

In [4]:
# !python biobertpt_contrastive_finetuning.py